# Pattern 1: Reflection

Reflection makes the agent reflect on its output. Or more generally, it makes multiple LLMs talk to each other in something like a **peer review process**. The reflection agent suggests modifications, additions, improvements in the writing style, and so on. This iterative process[^iterative] often leads to substantial gains in output quality, as the generation model benefits from external critique (i.e. different model, or just a different execution process[^reflection]). Reflection is ideal for high-stakes, critical tasks where reflection acts as a cheap QA step.

[^iterative]: The cost of running the model 2-3 times is far less than the cost of a significant error. For situations where the cost of a mistake is negligible, then using reflection isn't a good tradeoff.

[^reflection]: The reflection model is focused on evaluation, error detection, factual verification, or alignment with constraints. So it has a more specific goal than generating content from scratch.

![**Reflection Pattern**. Two LLMs iteratively improve the generated response through an iterative review process.](./img/pattern-reflective.png){#fig-pattern-reflective}

## Reflection steps

**Inference client.** Initializing the client for LLM inference and loading the API keys:

In [1]:
import pandas as pd

from openai import OpenAI
from notebooks.utils import load_dotenv, print
from IPython.display import display_markdown

load_dotenv(verbose=True)
client = OpenAI()

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


### System prompts

We will create two separate chat histories, one for generation and another for reflection. We set the generation system prompt as a developer tasked to write high-quality Python code. On the other hand, we set the reflection system prompt such that it only responds with feedback instead of rewriting the whole thing. Finally, we instruct the reflection agent to write `APPROVED` when satisfied so we can terminate the loop.

In [2]:
STOP_WORD = "APPROVED"

BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = f"""
You are tasked with generating critique and recommendations on the user's generated content. 
Your role is to help the user improve by pointing out strengths, weaknesses, and opportunities 
for refinement. 

You must NEVER provide full solutions, rewritten versions of the content, or long verbatim outputs. 
You may use short illustrative examples (1-3 lines or a single sentence) only when necessary to clarify 
a point. Providing a complete solution is a policy violation. 

If the user content has something wrong or something to be improved, output ONLY a clear list of 
recommendations and critiques. 

If you are satisfied and have no further strong recommendations, output EXACTLY the single word:

{STOP_WORD}

GUIDELINES FOR CRITIQUE:
- Forbidden Example: Rewriting the entire essay, code, or design for the user.
- Forbidden Example: Giving the full, corrected version of the user's work.
- Allowed Example: "Consider clarifying your thesis statement, e.g., make it one clear sentence."
- Good Example: Pointing out issues, suggesting improvements, or giving high-level recommendations without completing the work for the user.

GUIDELINES FOR APPROVAL:
- You must be fully satisfied with the content before approving.
- You must have checked that all past issues have been fully addressed.
- You must be sure there are no remaining issues, weaknesses, or areas for improvement.
- "{STOP_WORD}" must appear alone on a line, with no emojis, punctuation, or explanations.
- Do not mix "{STOP_WORD}" with any feedback or comments.
- Forbidden Example: "{STOP_WORD}, but consider improving your introduction."
- Good Example: "{STOP_WORD}"
"""

SHARED_DEFINITION_OF_DONE = """
DEFINITION OF DONE: The best solution is the SIMPLEST correct implementation that:
- SOLVES THE USER'S SPECIFIC PROBLEM COMPLETELY AND APPROPRIATELY
- Is readable and maintainable for the intended use case
- Avoids unnecessary complexity, over-engineering, or premature optimization  
- Uses the appropriate level of robustness (not necessarily maximal robustness)
- Prioritizes clarity and understandability over cleverness
- Delivers exactly what the user needs, nothing more and nothing less

KEY PRINCIPLE: The solution should be as simple as possible, but no simpler. 
It must address the user's actual needs while avoiding gold-plating.
"""

CODE_GENERATION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer tasked with generating high quality Python code.
Generate exactly one Python implementation that prioritizes SIMPLICITY, READABILITY, and PRACTICALITY.
Aim for the simplest correct solution that solves the problem without over-engineering.
Avoid unnecessary complexity, clever tricks, or advanced features unless absolutely necessary.
Do not provide multiple options, explanations, or alternative approaches.
Output only the final code in a fenced Python block.
""", SHARED_DEFINITION_OF_DONE, BASE_GENERATION_SYSTEM_PROMPT])

CODE_REFLECTION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer and strict code reviewer.

USER'S ORIGINAL REQUEST:
{user_prompt}

**Consider BOTH the user's specific needs AND our quality standards:**                                           

Your goal is to produce the simplest correct solution possible.
Avoid unnecessary complexity, clever tricks, or over-engineering.
Prioritize readability, maintainability, and clarity over novelty.

FORMAT REQUIREMENT:
- You MUST format your feedback in a **Markdown table** with the columns: | Issue | Details | Recommendation |
- Each row should contain exactly one critique and its corresponding recommendation.
- Do not use bullet points, numbered lists, or plain text for critiques — only a Markdown table.
                                           
Providing a complete solution is a policy violation. 
Forbidden Example (DO NOT DO THIS): Providing a full class or function rewrite. 
Your role is to help the user learn by giving feedback, not by coding for them. 
Allowed Example: “Consider validating input type, e.g., `if not isinstance(n, int): ...` ”

""", SHARED_DEFINITION_OF_DONE, BASE_REFLECTION_SYSTEM_PROMPT])

:::{.callout-caution}
Tuning the prompts took the most time / effort during the writing of this section. (ᵕ—ᴗ—) What worked for me: adding a **shared definition of done**, and having similar goals for both agents. In theory, having divergent goals can be good, but in practice it lead to agents going off-track, or getting into add-remove cycles. Or one agent dominating the other. Finally, the [reflection agent's system prompt]{.underline} include the **user prompt** to ground the agent's analysis in the specific context and intent of the user's request, while still maintaining the required quality standards.

:::

### Model choice

For the current task (generating code for a simple function), we use:

In [3]:
GENERATION_MODEL = "gpt-4.1"
REFLECTION_MODEL = "o3"

| Role        | Focus                                   |
|-------------|------------------------------------------------------|
| **Generation** | Creativity, fluency, diverse output. Feedback incorporation.                 |
| **Reflection** | Evaluation, error detection, factual verification, constraint alignment |


From our experiments (and performing code review IRL), reviewing is a nontrivial task: following guidelines, spotting subtle issues, and enforcing consistency needs strong reasoning capacity and attention to detail. We generally had best results with a fairly strong [reasoning model]{.underline} (e.g. `o3` and `gpt-oss`) as reflection model.

:::{.callout-tip}
The simplest setup is (1) to use the **same model** for generation and reflection. Another approach is (2) to use a **stronger reflection model**. This makes sense for tasks where it's easier to write than to critique, like summarization, and allows us to reduce generation cost. A good baseline is (3) a **large generator** with **medium-sized reflector**. This can miss subtle errors but is a cost-effective approach. Finally, we can (4) [fine-tune]{.underline} a **small, specialized reflector**. This can outperform generalists at specific feedback. The reflector should generally be a [reasoning model]{.underline} when dealing with complex problem solving, planning, logical critique, and multi-step reasoning. For optimizing creativity and style, the reflector can be a general-purpose completion model.
:::

### Generation step

We now ask the LLM to write an implementation of the Fibonacci sequence. Since it's only used for a quick demo, we expect the agents to converge to a simple solution. Pushing user prompt to generation agent:

In [4]:
from notebooks.agents.chat import ChatHistory, ChatCompletions

USER_PROMPT = """
Generate a Python implementation of merge sort. 
This will only be used for a quick demo. 
Don't worry too much about typing, just ensure it works correctly.
"""

# Initialize histories
generation_chat_history = ChatHistory(CODE_GENERATION_SYSTEM_PROMPT)
reflection_chat_history = ChatHistory(CODE_REFLECTION_SYSTEM_PROMPT.format(user_prompt=USER_PROMPT))

# Initialize generation with user prompt
generation_chat_history.update(role="user", prompt=USER_PROMPT)

**Initial version.** As usual, GA has role `assistant`. We send over the response to the RA with role `user`[^role].

[^role]: Agents acting in behalf of the user hence the `user` role.

In [5]:
completions = ChatCompletions(client)

code = completions.create(generation_chat_history, GENERATION_MODEL)
generation_chat_history.update(prompt=code, role="assistant")
reflection_chat_history.update(prompt=code, role="user")

:::{.callout-note collapse="false"}
## Initial generated code

In [6]:
#| echo: false
display_markdown(code, raw=True)

```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result
```

:::

### Reflection step

The generated critique is likewise sent over to the GA with `user` role.

In [7]:
review = completions.create(reflection_chat_history, REFLECTION_MODEL)
reflection_chat_history.update(prompt=review, role="assistant")
generation_chat_history.update(prompt=review, role="user")

:::{.callout-note collapse="false"}
## Feedback from reflection

In [8]:
#| echo: false
display_markdown(review, raw=True)

| Issue | Details | Recommendation |
|-------|---------|---------------|
| Missing input validation | The function assumes `arr` is an index-able sequence; passing `None` or a non-sequence raises a runtime error. | Guard with a lightweight check such as `if not isinstance(arr, Sequence): raise TypeError(...)` or convert the input to `list(arr)` when appropriate. |
| No documentation | Readers have to infer purpose, return type, and whether the sort is in-place. | Add a concise docstring describing what the function does, its parameters, return value, and complexity. |
| Unnecessary list slicing | Each recursive call copies sub-lists via `arr[:mid]` and `arr[mid:]`, increasing memory use and runtime constants. | For larger inputs or teaching efficiency, pass index boundaries (`start`, `end`) or use iterators to avoid repeated copying. |
| Recursion depth risk | Deeply nested recursion (`len(arr) ≳ 1 000`) may hit Python’s default recursion limit and raise `RecursionError`. | Mention or guard against this (e.g., `sys.setrecursionlimit`) or note an iterative alternative for very large datasets. |
| Limited flexibility | The implementation only supports default comparison (`<`) and cannot sort based on a key function or reverse order. | Accept optional `key` and `reverse` parameters (mirroring `list.sort`) for broader applicability in demos without adding much complexity. |

:::

:::{.callout-tip}
Out of all models we've tested, only OpenAI `o-` models strictly followed the review format.

:::

**Histories.** Chat histories after the first exchange. Both histories store the **generation>reflection** steps (in that causal order), only with different role assignments. This will be followed by the next generation step, and we keep iterating until the stopping condition is triggered by the RA. From the following tables we see that the generation history should have [even max length]{.underline}, while the reflection history should have [odd max length]{.underline}.

In [9]:
pd.DataFrame(generation_chat_history)

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\ndef merge_sort(arr):\n if len(ar...
3,user,| Issue | Details | Recommendation |\n|-------...


In [10]:
pd.DataFrame(reflection_chat_history)

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,```python\ndef merge_sort(arr):\n if len(ar...
2,assistant,| Issue | Details | Recommendation |\n|-------...


<span style="display: block; margin-bottom: 0.5em;"> </span>

Moreover, observe that we get the correct role sequence for the generation agent: `system` > `user` > [`assistant` > `user`] (one cycle). Similarly, for the refection agent, its `system` > [`user` > `assistant`] (one cycle). These are invariants even when we reach max length and the messages are replaced since messages are replaced two at a time.

## Full implementation

Combining the discussion and observations in a single class:

In [11]:
class ReflectionAgent:
    def __init__(self, 
        client, 
        generation_model: str, 
        reflection_model: str,
        generation_system_prompt="",
        reflection_system_prompt="",
        shared_definition_of_done="",
    ):
        self.completions = ChatCompletions(client)
        self.gen_model = generation_model
        self.ref_model = reflection_model
        self.gen_prompt = "\n".join([generation_system_prompt, shared_definition_of_done, BASE_GENERATION_SYSTEM_PROMPT])
        self.ref_prompt = "\n".join([reflection_system_prompt, shared_definition_of_done, BASE_REFLECTION_SYSTEM_PROMPT])

    def run(self, user_prompt, max_iter=10, history_max_len=6) -> dict:
        """Iterate generation-reflection cycles until max_iter or `APPROVED` found in reflection."""
        assert history_max_len % 2 == 0, "history_max_len must be even"

        # See remarks below
        ref_prompt = self.ref_prompt.format(user_prompt=user_prompt)
        ref_history = ChatHistory(ref_prompt, max_len=history_max_len-1, fixed_n=1)
        gen_history = ChatHistory(self.gen_prompt, max_len=history_max_len, fixed_n=2)
        gen_history.update(prompt=user_prompt, role="user")

        for step in range(max_iter):

            # Generate and push to reflection history as user
            generation = self.completions.create(gen_history, self.gen_model)
            gen_history.update(prompt=generation, role="assistant")
            ref_history.update(prompt=generation, role="user")

            # Critique and push to generation history as user
            reflection = self.completions.create(ref_history, self.ref_model)
            ref_history.update(prompt=reflection, role="assistant")
            gen_history.update(prompt=reflection, role="user")

            if STOP_WORD in reflection:
                print("[Stop Sequence found. Stopping the reflection loop.]")
                break
        
        return {
            "generation": generation,
            "steps": step + 1,
            "generation_history": gen_history,
            "reflection_history": ref_history,
        }

:::{.callout-note}
Note that the user prompt is injected into the reflection prompt so that the reflection model aligns with user objectives. Then, the process starts with the user prompt pushed to the generation agent. Finally, agents on track in terms of quality standards with a shared [definition of done](https://www.atlassian.com/agile/project-management/definition-of-done). Generation history has [even]{.underline} length: 2 fixed prompts and 2 messages per iteration. On the other hand, reflection history has [odd]{.underline}, i.e. length of generation history minus 1 (only 1 fixed prompt).
:::

Running the process for a few iterations:

In [12]:
reflection_agent = ReflectionAgent(
    client=client,
    generation_model=GENERATION_MODEL,
    reflection_model=REFLECTION_MODEL,
    generation_system_prompt=CODE_GENERATION_SYSTEM_PROMPT,
    reflection_system_prompt=CODE_REFLECTION_SYSTEM_PROMPT,
    shared_definition_of_done=SHARED_DEFINITION_OF_DONE,
)

output = reflection_agent.run(user_prompt=USER_PROMPT)

[Stop Sequence found. Stopping the reflection loop.]


### Final comments

In [13]:
output["steps"]

3

Setting `history_max_len=6` means that the *last two* review-generation cycle is stored for the generation model to reference in its next generation step. Although here, it's approved so we don't go through another cycle.
Thus, (`history_max_len` - 2) / 2 is the number of past cycles the generation model can reference. Also, it's nice that the first user prompt is retained so it's like the last two generations are done *only* with the original user and system prompt in mind.

In [14]:
pd.DataFrame(output["generation_history"])

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\nfrom collections.abc import Sequenc...
3,user,| Issue | Details | Recommendation |\n|-------...
4,assistant,```python\nfrom collections.abc import Sequenc...
5,user,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


This is also the number of past cycles the reflection model references:

In [15]:
pd.DataFrame(output["reflection_history"])

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,```python\nfrom collections.abc import Sequenc...
2,assistant,| Issue | Details | Recommendation |\n|-------...
3,user,```python\nfrom collections.abc import Sequenc...
4,assistant,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


Checking out the last code version and the final critique from the reflection agent:

In [16]:
#| echo: false
display_markdown(output["reflection_history"][1]["content"], raw=True)

```python
from collections.abc import Sequence

def merge_sort(arr: list) -> list:
    """
    Perform a stable merge sort on the input list.
    
    Parameters:
        arr (list): A list of items supporting comparison operators.
    
    Returns:
        list: A new sorted list containing the same elements.
    
    Notes:
        - The original list is not modified.
        - Sorting is stable (equal elements retain original order).
        - For demo simplicity, input is type-checked but slicing is used,
          which increases memory usage. For large lists or best performance,
          consider an implementation that avoids slicing.
    """
    if not isinstance(arr, Sequence):
        raise TypeError("merge_sort expects a sequence (like a list or tuple).")
    if len(arr) <= 1:
        return list(arr)
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left: list, right: list) -> list:
    """
    Merge two sorted lists into a single sorted list (stable).
    
    Parameters:
        left (list): Sorted list.
        right (list): Sorted list.
    
    Returns:
        list: A merged, sorted list containing all elements.
    """
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        # If elements are not comparable, this raises TypeError as expected
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

if __name__ == "__main__":
    # Quick demo and tests
    cases = [
        [],
        [1],
        [2, 1],
        [1, 2, 3, 4, 5],
        [5, 4, 3, 2, 1],
        [4, 2, 5, 3, 4, 1],
        [3, 3, 3],
        [7, -2, 0, 2, 7, 4],
    ]
    for case in cases:
        sorted_case = merge_sort(case)
        assert sorted_case == sorted(case), f"Failed on input: {case}"
    print("All tests passed.")
```

In [17]:
#| echo: false
display_markdown(output["reflection_history"][2]["content"], raw=True)

| Issue | Details | Recommendation |
|-------|---------|---------------|
| Inconsistent type hints vs. runtime behavior | The signature declares `arr: list`, but the function accepts any `Sequence`; this may confuse static analyzers and readers. | Change the annotation to `Sequence` (or `Sequence[Any]`) and, if desired, annotate the return as `list` to reflect actual behavior. |
| Unintended acceptance of strings/bytes | `str` and `bytes` satisfy `Sequence`; sorting them character-by-character is rarely what users intend. | Add an explicit check (e.g., `isinstance(arr, (str, bytes))`) and raise a `TypeError` with a helpful message. |
| Potential recursion-depth limitation | Purely recursive approach can hit `RecursionError` on very large inputs (≈1,000+ elements depending on platform). | Mention this caveat in the docstring or comment, or note that an iterative/stack-based variant is preferable for very deep lists. |
| Minimal test coverage feedback | Assertions verify correctness but give no output on failure details beyond the first failing case. | Consider using Python’s `unittest` or at least printing both expected and actual values when an assertion fails for clearer diagnostics. |

The review actually makes sense. Moreover, the generation agent followed the recommendations resulting in an approval. See the diff and the final version below.

### Final output

Comparing the results to see the effect of reflection:

:::{.callout-note collapse="false"}
## Final approved output

In [18]:
#| echo: false
display_markdown(output["generation"], raw=True)

```python
from collections.abc import Sequence
from typing import Any, List

def merge_sort(arr: Sequence[Any]) -> List[Any]:
    """
    Perform a stable merge sort on the input sequence.

    Parameters:
        arr (Sequence[Any]): A sequence of items supporting comparison operators (not a str or bytes).

    Returns:
        List[Any]: A new sorted list containing the same elements.

    Notes:
        - The original sequence is not modified.
        - Sorting is stable (equal elements retain original order).
        - For demo simplicity, input is type-checked but slicing is used,
          which increases memory usage.
        - Recursion depth may be exceeded for very large inputs; 
          consider an iterative variant if needed.
    Raises:
        TypeError: If arr is not a sequence, or is a str/bytes.
    """
    if not isinstance(arr, Sequence) or isinstance(arr, (str, bytes)):
        raise TypeError("merge_sort expects a sequence (like list or tuple), not str/bytes.")
    if len(arr) <= 1:
        return list(arr)
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left: List[Any], right: List[Any]) -> List[Any]:
    """
    Merge two sorted lists into a single sorted list (stable).

    Parameters:
        left (List[Any]): Sorted list.
        right (List[Any]): Sorted list.

    Returns:
        List[Any]: A merged, sorted list containing all elements.
    """
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        # If elements are not comparable, this raises TypeError as expected
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

if __name__ == "__main__":
    # Quick demo and test coverage with basic feedback
    cases = [
        [],
        [1],
        [2, 1],
        [1, 2, 3, 4, 5],
        [5, 4, 3, 2, 1],
        [4, 2, 5, 3, 4, 1],
        [3, 3, 3],
        [7, -2, 0, 2, 7, 4],
    ]
    passed = 0
    for i, case in enumerate(cases):
        sorted_case = merge_sort(case)
        expected = sorted(case)
        if sorted_case == expected:
            passed += 1
        else:
            print(f"Test case #{i+1} FAILED:")
            print(f"  Input   : {case}")
            print(f"  Expected: {expected}")
            print(f"  Got     : {sorted_case}")
    if passed == len(cases):
        print("All tests passed.")
    else:
        print(f"{passed} out of {len(cases)} tests passed.")
```

:::

In [ ]:
#| echo: false
import difflib
from IPython.display import display_html

text1 = output["reflection_history"][1]["content"]
text2 = output["reflection_history"][3]["content"]
lines1 = text1.splitlines(keepends=True)
lines2 = text2.splitlines(keepends=True)

differ = difflib.HtmlDiff()
html_diff = differ.make_file(lines1, lines2, fromdesc="Original Text", todesc="Modified Text")
display_html(html_diff, raw=True)

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
 "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

 

 
 
 
 
 

 
 
 
 
 
 Original Text Modified Text 
 
 f 1 ```python f 1 ```python 
 2 from collections.abc import Sequence 2 from collections.abc import Sequence 
 n n 3 from typing import Any, List 
 3 4 
 n 4 def merge_sort(arr: list) -> list: n 5 def merge_sort(arr: Sequence[Any]) -> List[Any]: 
 5     """ 6     """ 
 n 6     Perform a stable merge sort on the input  li s t . n 7     Perform a stable merge sort on the input s equence . 
 7      8   
 8     Parameters: 9     Parameters: 
 n 9         arr (list): A list of items supporting comparison operators. n 10         arr (Sequence[Any]): A sequence of items supporting comparison operators (not a str or bytes). 
 10      11   
 11     Returns: 12     Returns: 
 n 12          l ist: A new sorted list containing the same elements. n 13          L ist [Any] : A new sorted list containing the same elements. 
 13      14   
 14     Notes: 15     Notes: 
 n 15         - The original  l is t is  not modified. n 16         - The original  sequence  is not modified. 
 16         - Sorting is stable (equal elements retain original order). 17         - Sorting is stable (equal elements retain original order). 
 17         - For demo simplicity, input is type-checked but slicing is used, 18         - For demo simplicity, input is type-checked but slicing is used, 
 n 18           which increases memory usage. For large lists or best performance, n 19           which increases memory usage. 
 19           consider an implementation that avoids slicing. 20         - Recursion depth may be exceeded for very large inputs;  
 21           consider an iterative variant if needed. 
 22     Raises: 
 23         TypeError: If arr is not a sequence, or is a str/bytes. 
 20     """ 24     """ 
 n 21     if not isinstance(arr, Sequence): n 25     if not isinstance(arr, Sequence) or isinstance(arr, (str, bytes)): 
 22         raise TypeError("merge_sort expects a sequence (like  a  list or tuple).") 26         raise TypeError("merge_sort expects a sequence (like list or tuple) , not str/bytes .") 
 23     if len(arr) <= 1: 27     if len(arr) <= 1: 
 24         return list(arr) 28         return list(arr) 
 25     mid = len(arr) // 2 29     mid = len(arr) // 2 
 26     left = merge_sort(arr[:mid]) 30     left = merge_sort(arr[:mid]) 
 27     right = merge_sort(arr[mid:]) 31     right = merge_sort(arr[mid:]) 
 28     return merge(left, right) 32     return merge(left, right) 
 29 33 
 n 30 def merge(left:  l ist, right:  l ist) ->  l ist: n 34 def merge(left:  L ist [Any] , right:  L ist [Any] ) ->  L ist [Any] : 
 31     """ 35     """ 
 32     Merge two sorted lists into a single sorted list (stable). 36     Merge two sorted lists into a single sorted list (stable). 
 n 33      n 37   
 34     Parameters: 38     Parameters: 
 n 35         left ( l ist): Sorted list. n 39         left ( L ist [Any] ): Sorted list. 
 36         right ( l ist): Sorted list. 40         right ( L ist [Any] ): Sorted list. 
 37      41   
 38     Returns: 42     Returns: 
 n 39          l ist: A merged, sorted list containing all elements. n 43          L ist [Any] : A merged, sorted list containing all elements. 
 40     """ 44     """ 
 41     result = [] 45     result = [] 
 42     i = j = 0 46     i = j = 0 
 43     while i < len(left) and j < len(right): 47     while i < len(left) and j < len(right): 
 44         # If elements are not comparable, this raises TypeError as expected 48         # If elements are not comparable, this raises TypeError as expected 
 45         if left[i] <= right[j]: 49         if left[i] <= right[j]: 
 46             result.append(left[i]) 50             result.append(left[i]) 
 47             i += 1 51             i += 1 
 48         else: 52         else: 
 49             result.append(right[j]) 53             result.append(right[j]) 
 50             j += 1 